# Traceprop inline-precond — Pythia-160M / SST-2 pilot

Run cells 1-6 top to bottom. Cell 4 is the 30-subset **pilot**; cell 5 is the gate that
decides whether to launch the full 200-subset x 5-seed campaign. Cell 7 (Part B, Pythia-1B
overhead) is separate — run it on its own only when you choose to.

Tracks `main` (latest fixes) during HF bring-up; re-pin to a frozen commit before the final campaign.


## Cell 1 — GPU check + mount Drive

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE")
assert any(g in gpu for g in ("A100", "L4")), (
    f"Need an A100 or L4 GPU, got {gpu!r}. "
    "Runtime -> Change runtime type -> GPU (A100 or L4).")

from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Clone repo at the pinned commit + install deps

In [ ]:
COMMIT = "main"
%cd /content
!rm -rf Traceprop && git clone -q https://github.com/AmitoVrito/Traceprop.git
%cd /content/Traceprop
!git checkout -q {COMMIT}   # main during HF bring-up; re-pin before final campaign
!git log --oneline -1

# Colab ships torch (CUDA), transformers, scipy, numpy. Add the rest.
!pip -q install "datasets>=2.14" "peft>=0.7" pytest
!pip -q install --ignore-requires-python logix-ai   # setup.py caps py<3.11; wheel is pure-python
!pip -q uninstall -y torchao   # Colab's torchao 0.10 breaks peft 0.20 LoRA dispatch; unused here
!pip -q install -e . --no-deps                        # traceprop, deps already present

import torch, transformers, peft, logix, scipy, numpy, datasets
print(f"torch {torch.__version__} | transformers {transformers.__version__} | "
      f"peft {peft.__version__} | logix {logix.__version__} | "
      f"datasets {datasets.__version__} | numpy {numpy.__version__} | scipy {scipy.__version__}")
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Cell 3 — Unit tests (stops if anything fails)

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest",
                    "tests/unit/test_lora_logging.py", "-q"])
assert r.returncode == 0, "UNIT TESTS FAILED - do not proceed."
print("unit tests passed")

## Cell 4 — PILOT: Pythia-160M, SST-2, seed 0, 30 subsets, n_train 1500

In [ ]:
import time, subprocess, sys, os, json

os.chdir('/content/Traceprop/experiments')
cmd = [sys.executable, "exp35_logix_lds.py",
       "--backend", "hf", "--model", "EleutherAI/pythia-160m", "--data", "sst2",
       "--device", "cuda", "--gpu_check", "", "--track", "0",
       "--factored", "--inline_precond", "--tune_logix", "--logix_permodule_damping",
       "--n_train", "1500", "--n_test", "100", "--n_subsets", "30",
       "--subset_frac", "0.5", "--epochs", "3", "--batch", "16",
       "--proj_dim", "512", "--lora_init", "pca", "--seed", "0",
       "--precond_damping_grid", "1e-6,1e-5,1e-4,1e-3,1e-2,1e-1,1e0,1e1",
       "--precond_val_frac", "0.3",
       "--out", "results/exp35_p160_sst2_pilot_seed0.json", "--force"]
t0 = time.time()
p = subprocess.run(cmd, capture_output=True, text=True)
mins = (time.time() - t0) / 60

# persist logs to Drive so a failure/disconnect never loses the traceback
os.makedirs("/content/drive/MyDrive/traceprop_results/", exist_ok=True)
open("/content/drive/MyDrive/traceprop_results/pilot_stderr.txt", "w").write(p.stderr)
open("/content/drive/MyDrive/traceprop_results/pilot_stdout.txt", "w").write(p.stdout)

if p.returncode != 0:
    print("EXIT", p.returncode, "- PILOT FAILED. STDERR tail (also saved to Drive/pilot_stderr.txt):\n")
    print("\n".join(p.stderr.splitlines()[-45:]))
    raise SystemExit("pilot run failed - paste the STDERR tail above")

print(f"pilot runtime: {mins:.1f} min")
print("\n".join(p.stdout.splitlines()[-15:]))

d = json.load(open("results/exp35_p160_sst2_pilot_seed0.json"))
print(f"\ntarget test acc: {d.get('target_test_acc')}")
keys = ["logix_dot", "logix_preconditioned", "logix_preconditioned_tuned",
        "traceprop_factored_kfac8_dot", "traceprop_factored_kfac8_inlineprecond",
        "traceprop_factored_kfac7_inlineprecond", "random"]
print(f"\n{'method':<44}{'full LDS':>18}{'held-out eval':>16}")
for k in keys:
    if k in d["lds"]:
        f = d["lds"][k]
        h = d.get("lds_heldout_eval", {}).get(k, {})
        print(f"{k:<44}{f['mean']:+.4f} +/- {f['std']:.4f}   {h.get('mean', float('nan')):+.4f}")


## Cell 5 — GATE: proceed to the full campaign, or stop?

In [ ]:
import json
d = json.load(open("/content/Traceprop/experiments/results/exp35_p160_sst2_pilot_seed0.json"))
lds = d["lds"]
methods = {k: v["mean"] for k, v in lds.items() if k != "random"}
best = max(methods, key=methods.get)
rand = lds.get("random", {}).get("mean", 0.0)
print(f"random LDS      : {rand:+.4f}")
print(f"best method     : {best} = {methods[best]:+.4f}")
print(f"inline (kfac8)  : {lds.get('traceprop_factored_kfac8_inlineprecond',{}).get('mean',float('nan')):+.4f}")
print(f"LogIX tuned     : {lds.get('logix_preconditioned_tuned',{}).get('mean',float('nan')):+.4f}")
print()
if methods[best] >= 0.2:
    print("PROCEED: signal is clearly above noise. Launch the 200-subset x 5-seed")
    print("campaign (gpu_replication_runbook.md, Part A1). NOTE: 30-subset gaps are")
    print("NOT directional -- only the full campaign decides inline-vs-LogIX.")
else:
    print("STOP: LDS near noise. Report back. Adjust the SETUP (epochs / task / n_train),")
    print("NOT the method.")

## Cell 6 — Persist results to Drive + download zip (survive disconnects)

In [ ]:
import os, glob, shutil
dst = "/content/drive/MyDrive/traceprop_results/"
os.makedirs(dst, exist_ok=True)
src_dir = "/content/Traceprop/experiments/results"
copied = 0
for f in glob.glob(src_dir + "/exp35_p160_*"):
    shutil.copy(f, dst); copied += 1
print(f"copied {copied} pilot file(s) to {dst}")

zip_path = shutil.make_archive("/content/p160_pilot_results", "zip", src_dir)
print("zip:", zip_path)
from google.colab import files
files.download(zip_path)

## Cell 7 — PART B (SEPARATE): Pythia-1B in-training overhead

Not part of the pilot. Run this on its own when you want the overhead number
(total = factored logging + inline covariance vs plain training; plus covariance-only
marginal). Uses `torch.cuda.synchronize()` and drops a warmup repeat internally.

In [ ]:
import subprocess, sys, os, shutil
os.chdir('/content/Traceprop/experiments')
r = subprocess.run([sys.executable, "exp35_inlineprecond_overhead_hf.py",
                    "--model", "EleutherAI/pythia-1b", "--device", "cuda",
                    "--track", "0", "--kfac", "8", "--steps", "200",
                    "--batch", "8", "--seq", "128", "--repeats", "3",
                    "--out", "results/exp35_inlineprecond_overhead_hf_pythia1b.json"])
assert r.returncode == 0, "overhead run failed"
os.makedirs("/content/drive/MyDrive/traceprop_results/", exist_ok=True)
shutil.copy("results/exp35_inlineprecond_overhead_hf_pythia1b.json",
            "/content/drive/MyDrive/traceprop_results/")
print("overhead result saved to Drive")